# Module 8: Link Analysis, PageRank & HITS

**Z3003 Data Mining: Scaling to Large Data (BS Core) · IIT Madras Zanzibar**

Power iteration with dead-end handling, spider-trap teleportation, and HITS hubs/authorities.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/inyalala/iitmz-teaching-site/blob/main/courses/dm-scale/labs/DM08_PageRank_Link_Analysis.ipynb)

Run cells top to bottom (`Runtime > Run all`). Every method is self-contained and verified with assertions.

## 1. PageRank Power Iteration with Dead-End & Spider-Trap Teleportation

Setting $r^{(t+1)} = \beta M r^{(t)} + \frac{1 - \beta}{N}\mathbf{1}$ (with dead-end columns set to $1/N$) guarantees convergence to a unique stationary distribution summing to $1$.

In [1]:
import numpy as np


def pagerank(adj, beta=0.85, tol=1e-10, max_iter=200):
    nodes = sorted(adj)
    n = len(nodes)
    idx = {u: i for i, u in enumerate(nodes)}
    M = np.zeros((n, n))
    for u, nbrs in adj.items():
        if nbrs:
            for v in nbrs:
                M[idx[v], idx[u]] += 1.0 / len(nbrs)
        else:
            M[:, idx[u]] = 1.0 / n
    r = np.full(n, 1.0 / n)
    for step in range(1, max_iter + 1):
        rn = beta * (M @ r) + (1.0 - beta) / n
        if np.sum(np.abs(rn - r)) < tol:
            return {u: float(rn[idx[u]]) for u in nodes}, step
        r = rn
    return {u: float(r[idx[u]]) for u in nodes}, max_iter


web = {"A": ["B", "C", "D"], "B": ["A", "D"], "C": ["A"], "D": ["D"]}
ranks_085, steps = pagerank(web, beta=0.85)
ranks_100, _ = pagerank(web, beta=1.0, max_iter=150)
print(f"beta=0.85 ({steps} steps):", {k: round(v, 4) for k, v in ranks_085.items()})
print("beta=1.00 (spider trap):", {k: round(v, 4) for k, v in ranks_100.items()})
assert abs(sum(ranks_085.values()) - 1.0) < 1e-8 and ranks_085["A"] > 0.05


beta=0.85 (46 steps): {'A': 0.1336, 'B': 0.0753, 'C': 0.0753, 'D': 0.7158}
beta=1.00 (spider trap): {'A': 0.0, 'B': 0.0, 'C': 0.0, 'D': 1.0}
